# Exploring the occupancy permit data (layer 24)

There is no data dictionary for this layer, so before writing the enrichment pipeline I checked the data for the decisions the next steps depend on:

1. **Load:** what's in the layer, and are the key columns complete?
2. **Which rows:** permit ID, the 2026 date rule, statuses, and how complete the layer is
3. **Permit types:** are the two types spelled as in the brief?
4. **Utilities:** where are DC Water, PEPCO and Washington Gas named, and how?
5. **Locations:** how many per permit, any duplicates, and are the points on the road?

Each section ends with the decision it led to.

In [53]:
import numpy as np
import pandas as pd
import requests

pd.set_option("display.width", 160)

LAYER_24 = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_APPS/DDOT_TOPS/MapServer/24"

## 1. Load the data

I download the whole layer with no filter, so nothing is hidden before I decide how to filter. The service hands out at most 2,000 rows per request, so the code asks page by page.

In [ ]:
# The layer's own description: column names, types and code lists
layer_info = requests.get(LAYER_24, params={"f": "json"}, timeout=60).json()

rows, offset = [], 0
while True:
    params = {
        "where": "1=1",               # no filter: everything
        "outFields": "*",
        "orderByFields": "OBJECTID",  # same order every time, so pages don't overlap
        "resultOffset": offset,
        "resultRecordCount": 2000,
        "f": "json",
    }
    page = requests.get(f"{LAYER_24}/query", params=params, timeout=60).json()
    if "error" in page:               # a bad query still comes back as "OK"
        raise RuntimeError(page["error"])
    rows += [feature["attributes"] for feature in page["features"]]
    if not page.get("exceededTransferLimit"):
        break
    offset += len(page["features"])

permits = pd.DataFrame(rows)

# Dates arrive as milliseconds since 1970 in UTC, so convert them to DC time.
# Coded columns get an extra "_label" column with the readable value.
for field in layer_info["fields"]:
    name = field["name"]
    if field["type"] == "esriFieldTypeDate":
        permits[name] = pd.to_datetime(permits[name], unit="ms", utc=True).dt.tz_convert("America/New_York")
    domain = field.get("domain") or {}
    if domain.get("type") == "codedValue":
        labels = {str(c["code"]): c["name"] for c in domain["codedValues"]}
        permits[name + "_label"] = permits[name].astype("string").str.removesuffix(".0").map(labels)

print(f"{len(permits):,} rows (one per permit location), {len(layer_info['fields'])} columns in the layer")

In [3]:
# How complete are the columns this task depends on?
key_columns = [
    "TrackingNumber", "PermitNumber", "ApplicationDate", "IssueDate", "ExpirationDate",
    "EventTypeDescription", "StatusDescription",
    "ApplicantCompany", "PermitteeName", "OwnerName",
    "WorkLocationFullAddress", "XCoord", "YCoord", "RoadwaySegID",
]
permits[key_columns].notna().mean().round(3).to_frame("share filled")

,share filled
TrackingNumber,1.000
PermitNumber,0.707
ApplicationDate,1.000
IssueDate,0.707
ExpirationDate,1.000
EventTypeDescription,1.000
StatusDescription,1.000
ApplicantCompany,1.000
PermitteeName,1.000
OwnerName,0.637


**Section 1:** the layer has 20,039 location rows. Everything the pipeline needs is fully filled: tracking number, dates, permit type, status, applicant and permittee names, address and coordinates. `PermitNumber` and `IssueDate` are both about 71% filled (section 2 explains why). `OwnerName` is filled on about two-thirds of rows. `RoadwaySegID` is always empty, so the layer can't tell us the road, and the road lookup in step 4 is needed.

## 2. Which rows: permit ID, dates, statuses, coverage

In [4]:
# Is PermitNumber blank exactly when there is no issue date?
pd.crosstab(permits["PermitNumber"].isna(), permits["IssueDate"].isna(),
            rownames=["PermitNumber blank"], colnames=["IssueDate blank"])

IssueDate blank,False,True
PermitNumber blank,,
False,14173,0
True,0,5866


In [5]:
# The brief: "Permit applications or issued permits should be equal to or after January 1, 2026"
start = pd.Timestamp("2026-01-01", tz="America/New_York")
applied_2026 = permits["ApplicationDate"] >= start
issued_2026 = permits["IssueDate"] >= start
not_issued = permits["IssueDate"].isna()

date_group = pd.Series("applied and issued before 2026", index=permits.index, name="date group")
date_group[applied_2026 & issued_2026] = "applied 2026, issued 2026"
date_group[applied_2026 & not_issued] = "applied 2026, not issued yet"
date_group[~applied_2026 & issued_2026] = "applied 2025, issued 2026"
date_group[~applied_2026 & not_issued] = "applied 2025, not issued yet"

two_types = permits["EventTypeDescription"].isin(["Construction Staging Area", "Mobile Crane Work Zone"])
pd.crosstab(date_group[two_types], permits.loc[two_types, "EventTypeDescription"],
            margins=True, margins_name="Total")

EventTypeDescription,Construction Staging Area,Mobile Crane Work Zone,Total
date group,,,
"applied 2025, issued 2026",7,0,7
"applied 2025, not issued yet",5,0,5
"applied 2026, issued 2026",3645,155,3800
"applied 2026, not issued yet",3286,101,3387
Total,6943,256,7199


In [6]:
# Statuses of the rows that pass the date rule
in_2026 = applied_2026 | issued_2026
permits.loc[two_types & in_2026, "StatusDescription"].value_counts().to_frame("rows")

,rows
StatusDescription,
Issued,1991
Permit Expired,1794
Approved (Pending Payment),1398
Not Paid,744
Assigned,706
Resubmit,340
Rejected,120
Withdrawn,75
Suspended,11


In [7]:
# Is the layer complete for 2026? Applications per month (each application counted once)
applications = permits.drop_duplicates("TrackingNumber")
print("Earliest expiration date in the whole layer:", applications["ExpirationDate"].min().date())
applications["ApplicationDate"].dt.strftime("%Y-%m").value_counts().sort_index().to_frame("applications")

Earliest expiration date in the whole layer: 2026-08-30


,applications
ApplicationDate,
2025-06,1
2025-08,3
2025-09,8
2025-10,1
2025-11,8
2025-12,7
2026-01,16
2026-02,64
2026-03,168


**Section 2**

- **Permit ID:** `PermitNumber` is blank on exactly the rows with no issue date (no exceptions), so it is only given once a permit is issued. `TrackingNumber` is always filled. → **Use `TrackingNumber` as the permit ID.**
- **Date rule:** reading the brief as "applied **or** issued on/after 1 Jan 2026" keeps 7,194 rows of the two types. That includes 7 applied for in late 2025 and issued in 2026. Only 5 rows (applied in 2025, still not issued) fall outside. → **Applied or issued in 2026.**
- **Statuses:** everything from Issued and Expired to Rejected (120), Withdrawn (75) and Revoked (3). The brief doesn't filter on status, and a rejected application is still an application. → **Keep all, and include `StatusDescription` in the output so it can be filtered.**
- **Coverage:** applications climb from 16 in January to over 5,000 in September, and no permit in the layer expired before 30 August 2026. The layer seems to drop permits a few weeks after they expire. → **The result covers 2026 permits still in the layer on the day the data is pulled. January–August is heavily undercounted. This is stated in the README, and the pull date is recorded in the output.**

## 3. Permit types

In [8]:
# Are the brief's two types spelled exactly like this? Any other type or free-text "Other" entry about staging or cranes?
type_counts = permits.loc[in_2026, "EventTypeDescription"].value_counts()
print("Construction Staging Area:", type_counts.get("Construction Staging Area", 0))
print("Mobile Crane Work Zone:   ", type_counts.get("Mobile Crane Work Zone", 0))

similar_types = [t for t in type_counts.index if "staging" in t.lower() or "crane" in t.lower()]
print("\nTypes mentioning staging or crane:", similar_types)

other = permits.loc[in_2026, "OtherEventType"].dropna()
print("\n'Other' entries mentioning staging or crane:")
print(other[other.str.contains("stag|crane", case=False)].value_counts().to_string())

Construction Staging Area: 6938
Mobile Crane Work Zone:    256

Types mentioning staging or crane: ['Construction Staging Area', 'Mobile Crane Work Zone']

'Other' entries mentioning staging or crane:
OtherEventType
Union Stage Events                    3
UHG VIP Meeting - SUV/MPD Staging     2
Crew staging area                     1
Crane truck parking                   1


**Section 3:** both types exist with exactly the brief's spelling, and no other type mentions staging or cranes. Only 2 free-text "Other" entries are close ("Crew staging area", "Crane truck parking"). Free text isn't a reliable category, and the brief names the two types. → **Filter on the two exact names.**

Worth knowing: utilities also work under other permit types. For example, "Lead Free DC" is DC Water's lead pipe programme (1,713 rows in 2026). These are out of scope for this request, and the README mentions them.

## 4. Utilities

There is no utility column, so the utility has to come from the company names. In section 1 the applicant was sometimes a permit agent filing for another company, so I search all three name columns.

In [9]:
in_scope = permits[two_types & in_2026].copy()

patterns = {
    "DC Water": r"dc ?water|wasa|water and sewer|water & sewer",  # "dc ?water" also catches "DCWater/CIP"
    "PEPCO": r"pepco|potomac electric",
    "Washington Gas": r"washington gas|wash gas|wgl",
}
name_columns = ["ApplicantCompany", "PermitteeName", "OwnerName"]

# Every name matched, per utility and column (one count per application)
one_per_application = in_scope.drop_duplicates("TrackingNumber")
for utility, pattern in patterns.items():
    print(f"\n##### {utility} #####")
    for col in name_columns:
        hits = one_per_application[col].str.contains(pattern, case=False, na=False)
        if hits.any():
            print(f"  {col}: " + ", ".join(f"{name} ({n})" for name, n in one_per_application.loc[hits, col].value_counts().items()))


##### DC Water #####
  ApplicantCompany: DCWater/CIP (1087), DCWater/OPS (456), DCWater (1)
  PermitteeName: DCWater/OPS (301), DCWATER/OPS (155), DCWater/CIP (119), DC Water (29)
  OwnerName: DC Water (775), DCWATER/OPS (155), DCWater/OPS (147), DCWater/CIP (33)

##### PEPCO #####
  ApplicantCompany: Potomac Electric Power Company (620), Pepco (27)
  PermitteeName: Potomac Electric Power Company (411), POTOMAC ELECTRIC POWER COMPANY (46), Pepco Holdings Inc (30), Pepco (11), Pepco Holdings Inc. (6), Pepco holdings inc (1), Potomac Electric Power Company Potomac Electric Power Company (1)
  OwnerName: Potomac Electric Power Company (349), POTOMAC ELECTRIC POWER COMPANY (45), Pepco Holdings Inc (27), PEPCO INC (6), Pepco Holdings Inc. (6), Pepco Inc (6), Pepco holdings inc (1), Potomac Electric Power Company Potomac Electric Power Company (1), Pepco Holdings c/o AaRae Alexander (1), PEPCO (1)

##### Washington Gas #####
  ApplicantCompany: Wash Gas & Light Co. (911)
  PermitteeName: WA

In [10]:
# Tag each row with the utility named in any of the three columns
all_names = in_scope[name_columns].fillna("").agg(" | ".join, axis=1)
matches = pd.DataFrame({u: all_names.str.contains(p, case=False) for u, p in patterns.items()})
print("Rows naming more than one utility:", (matches.sum(axis=1) > 1).sum())

in_scope["Utility"] = matches.idxmax(axis=1).where(matches.any(axis=1))

summary = pd.DataFrame({
    "applications": in_scope.drop_duplicates("TrackingNumber")["Utility"].value_counts(),
    "locations": in_scope["Utility"].value_counts(),
})
summary.loc["Total"] = summary.sum()
summary

Rows naming more than one utility: 0


,applications,locations
Utility,,
DC Water,1544,1887
Washington Gas,911,1666
PEPCO,647,1373
Total,3102,4926


In [11]:
# Checks: who does the work on utility permits, and did any utility slip through unmatched?
utility_apps = in_scope[in_scope["Utility"].notna()].drop_duplicates("TrackingNumber")
other_apps = in_scope[in_scope["Utility"].isna()].drop_duplicates("TrackingNumber")
print("Most common permittees on utility applications:")
print(utility_apps["PermitteeName"].value_counts().head(6).to_string())
print("\nMost common applicants that did NOT match a utility:")
print(other_apps["ApplicantCompany"].value_counts().head(8).to_string())

Most common permittees on utility applications:
PermitteeName
Fort Myer Construction Corporation    747
Potomac Electric Power Company        411
DCWater/OPS                           301
DONALD HILL                           258
WASH GAS & LIGHT CO.                  213
EDWIN LANE                            159

Most common applicants that did NOT match a utility:
ApplicantCompany
N/A                               907
Metropolitan Permit Services       77
DFM Development Services, LLC      69
Comcast                            67
Stokes Plans & Permitting          36
LNDC Inc                           31
Department of General Services     29
Skanska Flatiron Joint Venture     24


**Section 4**

- The utility is named as the **applicant** on every utility application. The permittee is often a contractor doing the work (Fort Myer Construction, Capitol Paving and others).
- Names vary a lot. DC Water appears as "DCWater/CIP" (capital projects) and "DCWater/OPS" (operations), PEPCO as "Potomac Electric Power Company" and "Pepco Holdings Inc.", and Washington Gas as "Wash Gas & Light Co.". Searching for the brief's exact names would miss most of them. My first DC Water pattern needed a space ("dc water") and missed over 1,500 "DCWater/…" applications until I listed what it matched.
- No row names two utilities. The unmatched applicants are other companies (permit agents with no utility named, Comcast, developers).
- → **A location is kept if any of the three name columns matches a utility pattern, and the output gets a clean `Utility` column.** Today the applicant column alone finds the same applications. Checking all three costs nothing and still works if a utility is only named as permittee or owner.
- Result: **3,102 applications and 4,926 locations.**

## 5. Locations

The output needs one row per permit location, and step 4 sends each location's coordinates to the road lookup service.

In [12]:
utility_rows = in_scope[in_scope["Utility"].notna()]
per_application = utility_rows.groupby("TrackingNumber").size()
print(f"Locations per application: median {per_application.median():.0f}, max {per_application.max()}")
pd.cut(per_application, [0, 1, 2, 5, 10, np.inf], labels=["1", "2", "3-5", "6-10", "more than 10"]).value_counts().sort_index().to_frame("applications")

Locations per application: median 1, max 16


,applications
1,2254
2,426
3-5,365
6-10,48
more than 10,9


In [13]:
# Duplicates: the same location listed twice on one application, and how many distinct points there are overall
same_address = utility_rows.duplicated(["TrackingNumber", "WorkLocationAddressID"]).sum()
same_point = utility_rows.duplicated(["TrackingNumber", "XCoord", "YCoord"]).sum()
distinct_points = len(utility_rows[["XCoord", "YCoord"]].drop_duplicates())
print("Same address twice on one application:", same_address)
print("Same coordinates twice on one application:", same_point)
print(f"Distinct points across all applications: {distinct_points:,} (of {len(utility_rows):,} rows)")

Same address twice on one application: 0
Same coordinates twice on one application: 2
Distinct points across all applications: 2,585 (of 4,926 rows)


In [14]:
# Are the points already on the road centreline? OnSegX/OnSegY look like the point snapped to the street.
gap_m = np.hypot(utility_rows["XCoord"] - utility_rows["OnSegX"], utility_rows["YCoord"] - utility_rows["OnSegY"])
print(f"Rows with OnSeg coordinates: {gap_m.notna().sum():,} of {len(utility_rows):,}")
print(f"Point exactly on OnSeg position: {(gap_m == 0).mean():.1%}")
gap_m.describe(percentiles=[0.5, 0.9, 0.99]).round(1).to_frame("distance to OnSeg point (m)")

Rows with OnSeg coordinates: 4,926 of 4,926
Point exactly on OnSeg position: 89.5%


,distance to OnSeg point (m)
count,4926.0
mean,2.2
std,7.0
min,0.0
50%,0.0
90%,8.5
99%,29.6
max,99.9


**Section 5**

- **Locations per permit:** most applications have one location (2,254 of 3,102). 848 have several, up to 16. → **The "highest class per permit" step matters for about a quarter of applications.**
- **Duplicates:** no application lists the same address twice. 2 rows share coordinates with another address on the same application. They are different addresses, so both are kept. → **No de-duplication needed.**
- **Repeated spots:** the 4,926 rows use only 2,585 distinct points, because the same blocks come up again on different applications. → **Step 4 looks up each distinct point once and reuses the answer.** That's about half the calls to the road service.
- **Points vs the street:** the layer also stores `OnSegX`/`OnSegY`, which look like the point moved onto the street. 89.5% of points are already exactly there, 90% are within 8.5 m and 99% within 30 m. The furthest is 100 m. → The road lookup should work well. **Proposed:** send the original `XCoord`/`YCoord` with a 50 m search distance, and record the distance to the matched road for every row so far-off matches can be flagged.

## Next step

The download code in section 1 will be needed again: for the road segments (layer 162), for the road lookup service, and in the final script. Rather than copy it, I moved it into a small reusable helper, [`step2_rest_client.py`](step2_rest_client.py), with retries added for the thousands of road lookups. The rules decided here are applied in `step3_permits.py`.